In [1]:
import argparse
import h5py
import numpy as np
import os
import sys

In [7]:
import os
import h5py
import numpy as np
import csv

In [14]:
h5_path = r"data.h5"
meme_path = r"H12CORE_meme_format.meme"
out_txt = r"footprint_dataset_with_motif.txt"
out_csv = r"footprint_dataset_with_motif.csv"
max_pairs = None    # for testing, e.g., 5
max_motifs = None   # for testing, e.g., 10

BASES = np.array(['A', 'C', 'G', 'T'])

def onehot_to_seq(arr):
    arr = np.asarray(arr)
    if arr.ndim != 2 or arr.shape[1] != 4:
        raise ValueError(f"Unexpected one-hot shape: {arr.shape}")
    idx = np.argmax(arr, axis=1)
    return ''.join(BASES[idx].tolist())

def reverse_complement(seq):
    return seq.translate(str.maketrans('ACGT', 'TGCA'))[::-1]

def read_meme_motif_names(meme_path):
    motif_names = []
    with open(meme_path, 'r') as fh:
        for line in fh:
            if line.strip().startswith('MOTIF'):
                parts = line.strip().split()
                if len(parts) >= 2:
                    motif_names.append(parts[1])
    return motif_names

for path in (h5_path, meme_path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"File not found: {path}")

motif_names = read_meme_motif_names(meme_path)
if max_motifs is not None:
    motif_names = motif_names[:max_motifs]
M = len(motif_names)
print(f"Using {M} motifs")

with h5py.File(h5_path, "r") as f, \
     open(out_txt, "w") as fd_txt, \
     open(out_csv, "w", newline='') as fd_csv:

    pos = f['positive']
    shuf = f['shuffled']
    raw = f['raw']

    P_total = pos.shape[1]        
    P = P_total // 2               
    if max_pairs is not None:
        P = min(P, max_pairs)

    csv_writer = csv.writer(fd_csv)
    csv_writer.writerow(["sequence", "label", "motif"])

    for j in range(P):
        fwd_idx = j * 2
        rev_idx = j * 2 + 1

        raw_seq = onehot_to_seq(np.array(raw[0, fwd_idx]))
        raw_rev = onehot_to_seq(np.array(raw[0, rev_idx]))

        fd_txt.write(f"{j}_raw_forward\t{raw_seq}\n")
        fd_txt.write(f"{j}_raw_reverse\t{raw_rev}\n")
        csv_writer.writerow([raw_seq, "forward_true", "RAW"])
        csv_writer.writerow([raw_rev, "reverse_true", "RAW"])

        for i in range(M):
            motif_name = motif_names[i]

            pos_seq_fwd = onehot_to_seq(np.array(pos[i, fwd_idx]))
            shuf_seq_fwd = onehot_to_seq(np.array(shuf[i, fwd_idx]))
            pos_seq_rev = onehot_to_seq(np.array(pos[i, rev_idx]))
            shuf_seq_rev = onehot_to_seq(np.array(shuf[i, rev_idx]))

            entries = [
                (pos_seq_fwd, "forward_true"),
                (shuf_seq_fwd, "forward_shuffled"),
                (pos_seq_rev, "reverse_true"),
                (shuf_seq_rev, "reverse_shuffled")
            ]

            for seq, label in entries:
                # TXT
                fd_txt.write(f"{j}_{motif_name}_{label}\t{seq}\n")
                # CSV
                csv_writer.writerow([seq, label, motif_name])

print(f"TXT dataset written to {out_txt}")
print(f"CSV dataset written to {out_csv}")

Using 1443 motifs
TXT dataset written to footprint_dataset_with_motif.txt
CSV dataset written to footprint_dataset_with_motif.csv
